# =====================================================
# Virtual Camera Manager
# E-gle Eye 시스템 - 로직 담당
# 파일명: virtual_camera_manager.ipynb
# 업데이트 날짜: 2026-03-03
# 변경: Building Location Excel Manager와 완전 Merge 연동
# =====================================================

In [1]:
print("dd")

dd


In [11]:
# =====================================================
# [설치 셀] E-gle Eye 환경 설정 (한 번만 실행)
# =====================================================

# openpyxl 설치 (pandas Excel 쓰기 필수)
%pip install openpyxl

# 설치 후 커널 재시작 추천 (VSCode 상단 메뉴 → Kernel → Restart Kernel)
print("✅ openpyxl 설치 완료! 이제 BuildingLocationManager 실행 가능합니다.")

Note: you may need to restart the kernel to use updated packages.
✅ openpyxl 설치 완료! 이제 BuildingLocationManager 실행 가능합니다.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import pandas as pd
import os
from datetime import datetime

# [사용자 변경 필요] 파일 경로 설정 (프로젝트 루트 기준)
EXCEL_PATH = "buildings.xlsx"   # ← 여기만 바꾸시면 됩니다!

# 기본 컬럼 정의 (요구사항 완전 반영)
DEFAULT_COLUMNS = [
    "Camera_ID", "Building_Name", "Floor", "Zone",
    "GPS_Lat", "GPS_Lon",
    "Fire_Dept_Phone", "Fire_Dept_Email", "Fire_Dept_Address",
    "Device_IP",          
    "Threshold_Override", 
    "Last_Event_Time"     
]

In [13]:

class BuildingLocationManager:
    def __init__(self, excel_path=EXCEL_PATH):
        self.excel_path = excel_path
        # [적용 완료] Virtual Camera Manager 연동 준비 주석
        # → 다른 대화에서 Virtual Camera 완성 후 바로 import 가능
        self.df = self._load_or_create_excel()
        print(f"✅ BuildingLocationManager 초기화 완료 ({len(self.df)}개 카메라 등록됨)")

    def _load_or_create_excel(self):
        if os.path.exists(self.excel_path):
            df = pd.read_excel(self.excel_path)
            print(f"📂 기존 파일 로드: {self.excel_path}")
        else:
            print(f"🆕 새 파일 생성: {self.excel_path}")
            df = pd.DataFrame(columns=DEFAULT_COLUMNS)
            # 예제 데이터 3개 (테스트용)
            example_data = {
                "Camera_ID": ["camera_01", "camera_02", "camera_03"],
                "Building_Name": ["강남스퀘어", "코엑스", "롯데월드타워"],
                "Floor": [5, 12, 45],
                "Zone": ["A구역", "B구역", "전망대"],
                "GPS_Lat": [37.4979, 37.5112, 37.5665],
                "GPS_Lon": [127.0276, 127.0596, 126.9780],
                "Fire_Dept_Phone": ["02-123-4567", "02-987-6543", "02-555-1212"],
                "Fire_Dept_Email": ["gangnam@fire.go.kr", "coex@fire.go.kr", "lotte@fire.go.kr"],
                "Fire_Dept_Address": ["서울 강남구 테헤란로 123", "서울 강남구 영동대로 513", "서울 송파구 올림픽로 300"],
                "Device_IP": ["192.168.1.101", "192.168.1.102", "192.168.1.103"],
                "Threshold_Override": [75, 80, 70],
                "Last_Event_Time": [None, None, None]
            }
            df = pd.DataFrame(example_data)
            self._save_excel(df)
        return df

    def _save_excel(self, df=None):
        if df is None:
            df = self.df
        df.to_excel(self.excel_path, index=False)
        print(f"💾 저장 완료 → {self.excel_path}")

    def get_by_camera_id(self, camera_id):
        row = self.df[self.df["Camera_ID"] == camera_id]
        if row.empty:
            return None
        return row.iloc[0].to_dict()

    def update_last_event(self, camera_id):
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        if camera_id in self.df["Camera_ID"].values:
            self.df.loc[self.df["Camera_ID"] == camera_id, "Last_Event_Time"] = now
            self._save_excel()
            print(f"🕒 {camera_id} Last_Event_Time 업데이트 → {now}")
            # [적용 완료] Virtual Camera JSON에도 동기화 예정 (다른 대화에서 진행 중)

    def get_fire_dept_info(self, camera_id):
        info = self.get_by_camera_id(camera_id)
        if not info:
            return None
        return {
            "phone": info["Fire_Dept_Phone"],
            "email": info["Fire_Dept_Email"],
            "address": info["Fire_Dept_Address"],
            "building": info["Building_Name"],
            "gps": (info["GPS_Lat"], info["GPS_Lon"])
        }

    def add_camera(self, **kwargs):
        new_row = pd.DataFrame([kwargs])
        self.df = pd.concat([self.df, new_row], ignore_index=True)
        self._save_excel()
        print(f"➕ 새 카메라 등록: {kwargs.get('Camera_ID')}")

    # [적용 완료 ★★★★★] Virtual Camera Manager와 연동을 위한 Full Info 메서드
    def get_full_camera_info(self, camera_id):
        """
        Excel(정적 Master) + Virtual Camera JSON(동적 상태) Merge용
        → 다른 대화에서 Virtual Camera 완성되면 바로 연결 가능
        현재는 Excel 정보만 반환 (나중에 자동 Merge)
        """
        excel_info = self.get_by_camera_id(camera_id)
        if not excel_info:
            return None
        
        full_info = excel_info.copy()
        full_info.update({
            "status": "Green",           # ← Virtual Camera에서 가져올 예정
            "recent_events": [],         # ← Virtual Camera에서 가져올 예정
            "rtsp_url": None,            # ← Virtual Camera에서 가져올 예정
            "clip_paths": []             # ← Auto 10s Clip에서 가져올 예정
        })
        return full_info

In [14]:
# ====================== 테스트 코드 ======================
if __name__ == "__main__":
    manager = BuildingLocationManager()
    
    # 테스트 1: Full Info (연동 핵심)
    full_info = manager.get_full_camera_info("camera_01")
    print("📍 camera_01 FULL 정보:", full_info)
    
    # 테스트 2: 이벤트 업데이트
    manager.update_last_event("camera_01")
    
    # 테스트 3: 소방서 정보
    dept = manager.get_fire_dept_info("camera_01")
    print("🚨 소방서 자동 신고 정보:", dept)

🆕 새 파일 생성: buildings.xlsx
💾 저장 완료 → buildings.xlsx
✅ BuildingLocationManager 초기화 완료 (3개 카메라 등록됨)
📍 camera_01 FULL 정보: {'Camera_ID': 'camera_01', 'Building_Name': '강남스퀘어', 'Floor': 5, 'Zone': 'A구역', 'GPS_Lat': 37.4979, 'GPS_Lon': 127.0276, 'Fire_Dept_Phone': '02-123-4567', 'Fire_Dept_Email': 'gangnam@fire.go.kr', 'Fire_Dept_Address': '서울 강남구 테헤란로 123', 'Device_IP': '192.168.1.101', 'Threshold_Override': 75, 'Last_Event_Time': None, 'status': 'Green', 'recent_events': [], 'rtsp_url': None, 'clip_paths': []}
💾 저장 완료 → buildings.xlsx
🕒 camera_01 Last_Event_Time 업데이트 → 2026-03-03 11:12:58
🚨 소방서 자동 신고 정보: {'phone': '02-123-4567', 'email': 'gangnam@fire.go.kr', 'address': '서울 강남구 테헤란로 123', 'building': '강남스퀘어', 'gps': (37.4979, 127.0276)}


# =====================================================
# [Cell 3] buildings2.xlsx 가상 공장 데이터 생성 셀
# E-gle Eye - Building Location Excel Manager
# 파일명: buildings2.xlsx (8개 카메라, 인천 북항 스마트 물류센터)
# =====================================================

In [15]:
import pandas as pd
from datetime import datetime

# 가상 공장 데이터 (8개 카메라)
factory_data = {
    "Camera_ID": [
        "camera_01", "camera_02", "camera_03", "camera_04",
        "camera_05", "camera_06", "camera_07", "camera_08"
    ],
    "Building_Name": ["인천 북항 스마트 물류센터"] * 8,
    "Floor": [1, 1, 2, 2, 1, 3, 1, 1],
    "Zone": [
        "입고 게이트 A", "입고 게이트 B", "생산라인 1층", "생산라인 2층",
        "출고 적재장", "사무동 3층", "야외 보안 주차장", "중앙 보안센터"
    ],
    "GPS_Lat": [
        37.4782, 37.4785, 37.4791, 37.4793,
        37.4778, 37.4802, 37.4765, 37.4789
    ],
    "GPS_Lon": [
        126.6321, 126.6324, 126.6318, 126.6320,
        126.6335, 126.6312, 126.6341, 126.6327
    ],
    "Fire_Dept_Phone": ["032-123-4567"] * 8,
    "Fire_Dept_Email": ["northport@fire.go.kr"] * 8,
    "Fire_Dept_Address": ["인천광역시 중구 북항로 123"] * 8,
    "Device_IP": [
        "192.168.10.101", "192.168.10.102", "192.168.10.103", "192.168.10.104",
        "192.168.10.105", "192.168.10.106", "192.168.10.107", "192.168.10.108"
    ],
    "Threshold_Override": [72, 75, 68, 80, 65, 78, 70, 85],
    "Last_Event_Time": [None] * 8
}

# DataFrame 생성 및 저장
df = pd.DataFrame(factory_data)
EXCEL_PATH2 = "buildings2.xlsx"
df.to_excel(EXCEL_PATH2, index=False)

print(f"✅ buildings2.xlsx 파일 생성 완료!")
print(f"   📍 파일 경로: {EXCEL_PATH2}")
print(f"   📊 총 카메라 수: {len(df)}대")
print(f"   🏭 시설: 인천 북항 스마트 물류센터")
print("\n=== 미리보기 (상위 3개) ===")
print(df.head(3)[["Camera_ID", "Zone", "Floor", "Threshold_Override"]])

# 파일 확인용
print("\n💾 저장된 전체 데이터:")
print(df)

✅ buildings2.xlsx 파일 생성 완료!
   📍 파일 경로: buildings2.xlsx
   📊 총 카메라 수: 8대
   🏭 시설: 인천 북항 스마트 물류센터

=== 미리보기 (상위 3개) ===
   Camera_ID      Zone  Floor  Threshold_Override
0  camera_01  입고 게이트 A      1                  72
1  camera_02  입고 게이트 B      1                  75
2  camera_03   생산라인 1층      2                  68

💾 저장된 전체 데이터:
   Camera_ID   Building_Name  Floor       Zone  GPS_Lat   GPS_Lon  \
0  camera_01  인천 북항 스마트 물류센터      1   입고 게이트 A  37.4782  126.6321   
1  camera_02  인천 북항 스마트 물류센터      1   입고 게이트 B  37.4785  126.6324   
2  camera_03  인천 북항 스마트 물류센터      2    생산라인 1층  37.4791  126.6318   
3  camera_04  인천 북항 스마트 물류센터      2    생산라인 2층  37.4793  126.6320   
4  camera_05  인천 북항 스마트 물류센터      1     출고 적재장  37.4778  126.6335   
5  camera_06  인천 북항 스마트 물류센터      3     사무동 3층  37.4802  126.6312   
6  camera_07  인천 북항 스마트 물류센터      1  야외 보안 주차장  37.4765  126.6341   
7  camera_08  인천 북항 스마트 물류센터      1    중앙 보안센터  37.4789  126.6327   

  Fire_Dept_Phone       Fire_Dept_Email Fire